# Read and plot the original source

Same complete `MOI_controlled` readout and plotting options as `read_atst.ipynb`. Source ranges and condition annotations remain explicit; no ATST file is read. Matplotlib plots the source values directly.


In [ ]:
from pathlib import Path
import math
import matplotlib.pyplot as plt
import pandas as pd

readout_id = 'MOI_controlled'
rows, curves = [], []

import csv
with Path("read_shared_data/sources/TableS5_growth curve MOI controlled.csv").open(encoding="utf-8-sig", newline="") as file:
    source_rows = list(csv.reader(file))
header, data = source_rows[0], source_rows[1:]
# Documented source alignment repair: sole orphan value belongs to final blank.
assert data[-1][0] == "" and data[-1][64] == "0.2358" and sum(bool(v) for v in data[-1]) == 1
assert data[-2][0] == "82199.99" and data[-2][67] == ""
data[-2][67] = data[-1][64]
data = data[:-1]
time = [float(row[0]) for row in data]
counts = {}
for column, label in enumerate(header[1:], 1):
    counts[label] = counts.get(label, 0) + 1
    phage, moi = label.split("_MOI", 1) if "_MOI" in label else ("", "")
    rows.append(dict(type="blank" if label == "blank" else "phage_exposed" if phage else "uninfected_control", phage_id=phage, moi={"1": "1", "_01": "0.1", "_001": "0.01", "": ""}[moi], replicate=str(counts[label])))
    curves.append(pd.Series([float(row[column]) if row[column] else None for row in data], index=time))

# Local curve IDs preserve source order; different time grids align without interpolation.
layout = pd.DataFrame(rows)
layout["curve_id"] = [f"curve_{i}" for i in range(1, len(rows) + 1)]
readings = pd.concat(curves, axis=1).sort_index()
readings.columns = layout["curve_id"]
readings = readings.rename_axis("Time").reset_index()


In [ ]:
panels = [(None, layout)]
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby(['phage_id', 'moi', 'type'], sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = " | ".join(str(value) for value in condition)
        values = readings[group["curve_id"]]
        for index, curve_id in enumerate(group["curve_id"]):
            ax.plot(
                readings["Time"], readings[curve_id], color=color, alpha=0.6,
                label=label if index == 0 else None,
            )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("od600")
    ax.set_title(readout_id)
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()


In [ ]:
panels = [(None, layout)]
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby(['phage_id', 'moi', 'type'], sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = " | ".join(str(value) for value in condition)
        values = readings[group["curve_id"]]
        median = values.median(axis=1)
        minimum = values.min(axis=1)
        maximum = values.max(axis=1)
        ax.errorbar(
            readings["Time"], median,
            yerr=[median - minimum, maximum - median],
            fmt="o-", capsize=3, color=color, label=label,
        )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("od600")
    ax.set_title(readout_id)
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()
